# Notebook 02 — Data Preprocessing
**Project:** Flight–Bird Strike Risk Prediction and Mitigation System  
**Author:** A.D.P.I. Gunawardhana (CS/2020/011)  

Pipeline:
1. Load raw FAA dataset (Public.xlsx)
2. Column selection
3. Target variable encoding (DAMAGE_LEVEL)
4. Handle missing values
5. Feature engineering (temporal features, species grouping)
6. Categorical encoding (One-Hot)
7. Train/test split and save

## 0. Environment Setup

In [ ]:
import sys, os, shutil, subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_DIR  = "/content/flight-bird-strike"
    DRIVE_RAW = "/content/drive/MyDrive/Flight-Bird-Strike-MyResearch/data/raw"

    # ── 1. Mount Google Drive FIRST ────────────────────────────────────────────
    from google.colab import drive
    drive.mount('/content/drive')

    # ── 2. Clone / update the GitHub repo ─────────────────────────────────────
    # If your repo is PRIVATE: paste your GitHub Personal Access Token below.
    # Generate at: github.com → Settings → Developer Settings → Personal Access Tokens (classic)
    # If PUBLIC: leave GITHUB_TOKEN = ""
    GITHUB_TOKEN = ""   # <-- paste token here if repo is private
    GITHUB_USER  = "PawanGunawardhana"
    GITHUB_REPO  = "flight-bird-strike"

    if GITHUB_TOKEN:
        clone_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
    else:
        clone_url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

    if not os.path.exists(REPO_DIR):
        result = subprocess.run(["git", "clone", clone_url, REPO_DIR], capture_output=True, text=True)
        if result.returncode != 0:
            print("ERROR: git clone failed. Is the repo private?")
            print("  -> Set GITHUB_TOKEN above with your Personal Access Token.")
            raise RuntimeError(result.stderr)
        print("Repo cloned successfully.")
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
        print("Repo updated.")

    # ── 3. Add cloned repo to Python path ─────────────────────────────────────
    sys.path.insert(0, REPO_DIR)

    # ── 4. Install dependencies ────────────────────────────────────────────────
    subprocess.run(["pip", "install", "-r", f"{REPO_DIR}/requirements.txt", "-q"], check=True)

    # ── 5. Copy datasets from Drive → repo's data/raw/ ────────────────────────
    os.makedirs(f"{REPO_DIR}/data/raw", exist_ok=True)
    for fname in ["Public.xlsx", "Bird_strikes.csv"]:
        dst = f"{REPO_DIR}/data/raw/{fname}"
        src = f"{DRIVE_RAW}/{fname}"
        if not os.path.exists(dst):
            if os.path.exists(src):
                shutil.copy(src, dst)
                print(f"  Copied {fname}")
            else:
                print(f"  WARNING: {src} not found on Drive. Check your Drive folder name.")
        else:
            print(f"  {fname} already in data/raw/, skipping copy")

else:
    sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from config import cfg
from src.data.loader import load_faa_full
from src.data.preprocessor import preprocess, DAMAGE_LEVEL_LABELS, get_feature_columns

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120})
print(f"\nEnvironment: {'Google Colab' if IN_COLAB else 'Local (VS Code)'}")
print(cfg)


## 1. Load Raw Data

In [ ]:
df_raw = load_faa_full()
print(f"Raw shape: {df_raw.shape}")
df_raw[['DAMAGE_LEVEL', 'INDICATED_DAMAGE', 'PHASE_OF_FLIGHT', 'SPECIES', 'SKY', 'HEIGHT', 'SPEED']].head(5)

## 2. Missing Value Analysis (Before Preprocessing)

In [ ]:
# Show columns with >30% missing
missing = (df_raw.isna().sum() / len(df_raw) * 100).sort_values(ascending=False)
high_missing = missing[missing > 30]

fig, ax = plt.subplots(figsize=(12, max(4, len(high_missing) * 0.3)))
high_missing[::-1].plot(kind='barh', ax=ax, color='salmon', edgecolor='black')
ax.set_title('Columns with > 30% Missing Values', fontweight='bold')
ax.set_xlabel('Missing %')
plt.tight_layout()
plt.savefig(f'{cfg.FIGURES_DIR}/missing_values.png', bbox_inches='tight')
plt.show()

print(f"\n{len(high_missing)} columns have >30% missing data")

## 3. Run Full Preprocessing Pipeline

In [ ]:
df_processed = preprocess(df_raw, save=True)
print(f"\nProcessed dataset shape: {df_processed.shape}")
df_processed.head(3)

## 4. Class Imbalance Check & Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Multi-class: damage_label
vc_multi = df_processed['damage_label'].value_counts().sort_index()
labels_multi = [DAMAGE_LEVEL_LABELS.get(k, str(k)) for k in vc_multi.index]
axes[0].bar(labels_multi, vc_multi.values, color=sns.color_palette('Set2'), edgecolor='black')
axes[0].set_title('Multi-class Target: damage_label', fontweight='bold')
axes[0].set_xlabel('Damage Level')
axes[0].set_ylabel('Count')
for i, v in enumerate(vc_multi.values):
    axes[0].text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)

# Binary: damage_binary
vc_bin = df_processed['damage_binary'].value_counts().sort_index()
axes[1].bar(['No Damage (0)', 'Damaged (1)'], vc_bin.values,
            color=['steelblue', 'coral'], edgecolor='black')
axes[1].set_title('Binary Target: damage_binary', fontweight='bold')
axes[1].set_ylabel('Count')
for i, v in enumerate(vc_bin.values):
    axes[1].text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f'{cfg.FIGURES_DIR}/class_balance.png', bbox_inches='tight')
plt.show()

imbalance_ratio = vc_bin.max() / vc_bin.min()
print(f"Binary class imbalance ratio: {imbalance_ratio:.1f}:1")
print("NOTE: We will use class_weight='balanced' in all models to handle this.")

## 5. Feature Overview After Encoding

In [ ]:
feature_cols = get_feature_columns(df_processed)
print(f"Total features: {len(feature_cols)}")
print("\nFirst 30 features:")
print(feature_cols[:30])

# Check for any remaining nulls
null_check = df_processed[feature_cols].isna().sum().sum()
print(f"\nRemaining null values in features: {null_check}")

## 6. Train / Test Split

In [ ]:
X = df_processed[feature_cols]
y = df_processed['damage_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=cfg.TEST_SIZE,
    random_state=cfg.RANDOM_STATE,
    stratify=y
)

# Save train and test sets
train_df = X_train.copy()
train_df['damage_label'] = y_train.values
train_df['damage_binary'] = df_processed.loc[X_train.index, 'damage_binary'].values
train_df.to_csv(cfg.TRAIN_CSV, index=False)

test_df = X_test.copy()
test_df['damage_label'] = y_test.values
test_df['damage_binary'] = df_processed.loc[X_test.index, 'damage_binary'].values
test_df.to_csv(cfg.TEST_CSV, index=False)

print(f"Train set: {len(train_df):,} rows → saved to {cfg.TRAIN_CSV}")
print(f"Test set:  {len(test_df):,} rows → saved to {cfg.TEST_CSV}")
print(f"\nTrain class distribution:")
print(y_train.value_counts().sort_index().rename(DAMAGE_LEVEL_LABELS))

## 7. Preprocessing Complete ✓

**Output files created:**
- `data/processed/faa_processed.csv` — full preprocessed dataset
- `data/processed/train.csv` — training split (80%)
- `data/processed/test.csv` — test split (20%)

**Next step:** Run `03_Model_Training.ipynb`

In [ ]:
# ── Save processed files to Google Drive for persistence across Colab sessions ──
# Colab wipes /content/ when a session ends. This cell backs up the 3 CSVs
# to your Drive so notebook 03 can restore them in a new session.

if IN_COLAB:
    DRIVE_PROCESSED = "/content/drive/MyDrive/Flight-Bird-Strike-MyResearch/data/processed"
    os.makedirs(DRIVE_PROCESSED, exist_ok=True)

    for fname in ["faa_processed.csv", "train.csv", "test.csv"]:
        src = f"{REPO_DIR}/data/processed/{fname}"
        dst = f"{DRIVE_PROCESSED}/{fname}"
        if os.path.exists(src):
            shutil.copy(src, dst)
            print(f"Backed up to Drive: {fname}")
        else:
            print(f"WARNING: {src} not found locally — was preprocessing run?")

    print("\nAll processed CSVs saved to Drive.")
    print(f"Location: {DRIVE_PROCESSED}")
else:
    print("Local environment — no Drive backup needed (files already in data/processed/).")
